# Projeto Python IA: Inteligência Artificial e Previsões

### Case: Score de Crédito dos Clientes

Você foi contratado por um banco para conseguir definir o score de crédito dos clientes. Você precisa analisar todos os clientes do banco e, com base nessa análise, criar um modelo que consiga ler as informações do cliente e dizer automaticamente o score de crédito dele: Ruim, Ok, Bom

### O Código a seguir é a minha proposta de solução para o problema

### Passo 1 - Importar Bibliotecas

In [1]:
import pandas as pd
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

### Passo 2 - Importar dados

In [2]:
tabela = pd.read_csv("clientes.csv")
display(tabela)
display(tabela.info())

,id_cliente,mes,idade,profissao,salario_anual,num_contas,num_cartoes,juros_emprestimo,num_emprestimos,dias_atraso,...,idade_historico_credito,investimento_mensal,comportamento_pagamento,saldo_final_mes,score_credito,emprestimo_carro,emprestimo_casa,emprestimo_pessoal,emprestimo_credito,emprestimo_estudantil
0,3392,1,23.0,cientista,19114.12,3.0,4.0,3.0,4.0,3.0,...,265.0,21.465380,alto_gasto_pagamento_baixos,312.494089,Good,1,1,1,1,0
1,3392,2,23.0,cientista,19114.12,3.0,4.0,3.0,4.0,3.0,...,266.0,21.465380,baixo_gasto_pagamento_alto,284.629162,Good,1,1,1,1,0
2,3392,3,23.0,cientista,19114.12,3.0,4.0,3.0,4.0,3.0,...,267.0,21.465380,baixo_gasto_pagamento_medio,331.209863,Good,1,1,1,1,0
3,3392,4,23.0,cientista,19114.12,3.0,4.0,3.0,4.0,5.0,...,268.0,21.465380,baixo_gasto_pagamento_baixo,223.451310,Good,1,1,1,1,0
4,3392,5,23.0,cientista,19114.12,3.0,4.0,3.0,4.0,6.0,...,269.0,21.465380,alto_gasto_pagamento_medio,341.489231,Good,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,37932,4,25.0,mecanico,39628.99,4.0,6.0,7.0,2.0,23.0,...,378.0,24.028477,alto_gasto_pagamento_alto,479.866228,Poor,1,0,0,0,1
99996,37932,5,25.0,mecanico,39628.99,4.0,6.0,7.0,2.0,18.0,...,379.0,24.028477,alto_gasto_pagamento_medio,496.651610,Poor,1,0,0,0,1
99997,37932,6,25.0,mecanico,39628.99,4.0,6.0,7.0,2.0,27.0,...,380.0,24.028477,alto_gasto_pagamento_alto,516.809083,Poor,1,0,0,0,1
99998,37932,7,25.0,mecanico,39628.99,4.0,6.0,7.0,2.0,20.0,...,381.0,24.028477,baixo_gasto_pagamento_alto,319.164979,Standard,1,0,0,0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id_cliente                100000 non-null  int64  
 1   mes                       100000 non-null  int64  
 2   idade                     100000 non-null  float64
 3   profissao                 100000 non-null  object 
 4   salario_anual             100000 non-null  float64
 5   num_contas                100000 non-null  float64
 6   num_cartoes               100000 non-null  float64
 7   juros_emprestimo          100000 non-null  float64
 8   num_emprestimos           100000 non-null  float64
 9   dias_atraso               100000 non-null  float64
 10  num_pagamentos_atrasados  100000 non-null  float64
 11  num_verificacoes_credito  100000 non-null  float64
 12  mix_credito               100000 non-null  object 
 13  divida_total              100000 non-null  fl

None

### Passo 3 - Tratar os dados

In [3]:
# Valores nulos
tabela = tabela.dropna()

# Valores duplicados
tabela = tabela.drop_duplicates()

# Removendo a coluna ID
if "id_cliente" in tabela.columns:
    tabela = tabela.drop("id_cliente", axis=1)

# Codificação de variáveis categóricas
encoders = {}
for coluna in tabela.columns:
    if tabela[coluna].dtype == 'object':
        encoders[coluna] = LabelEncoder()
        tabela[coluna] = encoders[coluna].fit_transform(tabela[coluna])
        display(encoders[coluna])
        
display(tabela)
display(tabela.info())

LabelEncoder()

LabelEncoder()

LabelEncoder()

LabelEncoder()

,mes,idade,profissao,salario_anual,num_contas,num_cartoes,juros_emprestimo,num_emprestimos,dias_atraso,num_pagamentos_atrasados,...,idade_historico_credito,investimento_mensal,comportamento_pagamento,saldo_final_mes,score_credito,emprestimo_carro,emprestimo_casa,emprestimo_pessoal,emprestimo_credito,emprestimo_estudantil
0,1,23.0,2,19114.12,3.0,4.0,3.0,4.0,3.0,7.0,...,265.0,21.465380,1,312.494089,0,1,1,1,1,0
1,2,23.0,2,19114.12,3.0,4.0,3.0,4.0,3.0,4.0,...,266.0,21.465380,3,284.629162,0,1,1,1,1,0
2,3,23.0,2,19114.12,3.0,4.0,3.0,4.0,3.0,7.0,...,267.0,21.465380,5,331.209863,0,1,1,1,1,0
3,4,23.0,2,19114.12,3.0,4.0,3.0,4.0,5.0,4.0,...,268.0,21.465380,4,223.451310,0,1,1,1,1,0
4,5,23.0,2,19114.12,3.0,4.0,3.0,4.0,6.0,4.0,...,269.0,21.465380,2,341.489231,0,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,4,25.0,11,39628.99,4.0,6.0,7.0,2.0,23.0,7.0,...,378.0,24.028477,0,479.866228,1,1,0,0,0,1
99996,5,25.0,11,39628.99,4.0,6.0,7.0,2.0,18.0,7.0,...,379.0,24.028477,2,496.651610,1,1,0,0,0,1
99997,6,25.0,11,39628.99,4.0,6.0,7.0,2.0,27.0,6.0,...,380.0,24.028477,0,516.809083,1,1,0,0,0,1
99998,7,25.0,11,39628.99,4.0,6.0,7.0,2.0,20.0,6.0,...,381.0,24.028477,3,319.164979,2,1,0,0,0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   mes                       100000 non-null  int64  
 1   idade                     100000 non-null  float64
 2   profissao                 100000 non-null  int32  
 3   salario_anual             100000 non-null  float64
 4   num_contas                100000 non-null  float64
 5   num_cartoes               100000 non-null  float64
 6   juros_emprestimo          100000 non-null  float64
 7   num_emprestimos           100000 non-null  float64
 8   dias_atraso               100000 non-null  float64
 9   num_pagamentos_atrasados  100000 non-null  float64
 10  num_verificacoes_credito  100000 non-null  float64
 11  mix_credito               100000 non-null  int32  
 12  divida_total              100000 non-null  float64
 13  taxa_uso_credito          100000 non-null  fl

None

### Passo 4 - Particionar as colunas entre Objetivo e Parâmetros

In [4]:
y = tabela['score_credito'] # Variável alvo

x = tabela.drop('score_credito', axis=1) # Variáveis independentes

### Passo 5 - Particionar os dados entre Treino e Teste

In [5]:
# Divisão dos dados em treino e teste sem estratificação
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Estratificação para manter a proporção das classes
x_train_s, x_test_s, y_train_s, y_test_s = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# 1. MANTER O PADRÃO DE CORES E ORDEM
ordem_categorias = [0,1,2]
cores = ["#873C2F", "#55B144", "#29407D"] # Mesma ordem das categorias acima

# 2. FUNÇÃO PARA GERAR OS GRÁFICOS LADO A LADO
def comparar_com_original(dados_alvo, titulo_alvo):
    
    # Contar os valores forçando a ordem exata das categorias (preenche com 0 se a classe não existir)
    contagem_original = y.value_counts().reindex(ordem_categorias).fillna(0)
    contagem_alvo = dados_alvo.value_counts().reindex(ordem_categorias).fillna(0)

    # Criar a figura com 2 gráficos (1 linha, 2 colunas)
    fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'pie'}]],
                        subplot_titles=['Distribuição Original', titulo_alvo])

    # Gráfico 1 (Esquerda) - Distribuição Original
    fig.add_trace(go.Pie(
        labels=encoders['score_credito'].inverse_transform(ordem_categorias), values=contagem_original.values, 
        marker=dict(colors=cores), hole=0.4, 
        sort=False, # <-- ISSO GARANTE QUE AS FATIAS NÃO MUDEM DE LUGAR
        textinfo='percent+label', name='Original'
    ), row=1, col=1)

    # Gráfico 2 (Direita) - Dados Selecionados (Treino ou Teste)
    fig.add_trace(go.Pie(
        labels=encoders['score_credito'].inverse_transform(ordem_categorias), values=contagem_alvo.values, 
        marker=dict(colors=cores), hole=0.4, 
        sort=False, # <-- ISSO GARANTE QUE AS FATIAS NÃO MUDEM DE LUGAR
        textinfo='percent+label', name=titulo_alvo
    ), row=1, col=2)

    # Atualizar o layout: Fundo transparente e legenda padronizada
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', # Fundo fora do gráfico transparente
        plot_bgcolor='rgba(0,0,0,0)',  # Fundo do gráfico transparente
        legend_title='Score de Crédito',
        font_color='gray',
        legend=dict(
            orientation='h', 
            yanchor='top', y=-0.1, 
            xanchor='center', x=0.5
        )
    )
    
    fig.show()

# --- GERAR TODOS OS GRÁFICOS ---
# O Gráfico Original aparecerá sempre à esquerda de cada um destes:

# Comparação 1: Treino Sem Estratificação
comparar_com_original(y_train, 'Treino - Sem Estratificação')

# Comparação 2: Teste Sem Estratificação
comparar_com_original(y_test, 'Teste - Sem Estratificação')

# Comparação 3: Treino Com Estratificação
comparar_com_original(y_train_s, 'Treino Estratificado')

# Comparação 4: Teste Com Estratificação
comparar_com_original(y_test_s, 'Teste Estratificado')

### Passo 6 - Normalizar Dados

In [6]:
scaler = StandardScaler()
x_train_norm = scaler.fit_transform(x_train)
x_test_norm = scaler.transform(x_test)

### Passo 7 - Escolher modelos

In [7]:
# Modelos sem normalização e estratificacao
arvore = RandomForestClassifier()
modelo_xgb = XGBClassifier()

# Modelos com dados normalizados
arvore_norm = RandomForestClassifier()
modelo_xgb_norm = XGBClassifier()

# Modelos com dados estratificados
arvore_s = RandomForestClassifier()
modelo_xgb_s = XGBClassifier()

# Instanciar o Transformer (TabNet)
# Ao contrário das árvores, as redes neuronais precisam de "otimizadores" para aprender
modelo_tabnet = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax', # Mecanismo de atenção da rede
    verbose=1
)

modelo_tabnet_norm = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax', # Mecanismo de atenção da rede
    verbose=1
)

modelo_tabnet_s = TabNetClassifier(
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax', # Mecanismo de atenção da rede
    verbose=1
)

c:\Users\jluck\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning:

Device used : cpu



### Passo 8 - Treinar os modelos

In [8]:
# Treinamento dos modelos sem normalização
arvore.fit(x_train, y_train)
modelo_xgb.fit(x_train, y_train)

# Treinamento dos modelos com normalização
arvore_norm.fit(x_train_norm, y_train)
modelo_xgb_norm.fit(x_train_norm, y_train)

# Treinamento dos modelos com dados estratificados
arvore_s.fit(x_train_s, y_train_s)
modelo_xgb_s.fit(x_train_s, y_train_s)

# 2. Treinar o modelo
# As redes neuronais treinam em "Épocas" (passam pelos dados várias vezes).
# O TabNet gosta de receber os dados de teste logo no treino para ir medindo 
# a acurácia a cada passo e parar sozinho antes de "decorar" os dados (overfitting).

print("A iniciar o treino do Transformer TabNet...")

epocas = 200

# --- 1. MODELO BASE (SEM ESTRATIFICAÇÃO / SEM NORMALIZAÇÃO) ---
modelo_tabnet.fit(
    X_train=x_train.values,
    y_train=y_train.values, 
    eval_set=[(x_train.values, y_train.values), (x_test.values, y_test.values)], 
    eval_name=['Treino', 'Validacao'],
    eval_metric=['accuracy'],
    max_epochs=epocas, 
    patience=10,   
    batch_size=256, virtual_batch_size=128 
)

# --- 2. MODELO COM NORMALIZAÇÃO (SEM ESTRATIFICAÇÃO) ---
# Nota: x_train_norm e x_test_norm já saíram do StandardScaler como matrizes puras, 
# então NÃO precisam de .values (mas os y sim!)
modelo_tabnet_norm.fit(
    X_train=x_train_norm, 
    y_train=y_train.values, 
    eval_set=[(x_train_norm, y_train.values), (x_test_norm, y_test.values)], 
    eval_name=['Treino', 'Validacao'],
    eval_metric=['accuracy'],
    max_epochs=epocas, 
    patience=10,   
    batch_size=256, virtual_batch_size=128 
)

# --- 3. MODELO COM ESTRATIFICAÇÃO (E COM NORMALIZAÇÃO, CASO TENHA APLICADO) ---
# Certifique-se se usou x_train_s normalizado ou bruto. Se for bruto DataFrame, use .values.
# Assumindo que x_train_s e x_test_s são DataFrames:
modelo_tabnet_s.fit(
    X_train=x_train_s.values,
    y_train=y_train_s.values, 
    eval_set=[(x_train_s.values, y_train_s.values), (x_test_s.values, y_test_s.values)],
    eval_name=['Treino', 'Validacao'],
    eval_metric=['accuracy'],
    max_epochs=epocas, 
    patience=10,   
    batch_size=256, virtual_batch_size=128 
)

A iniciar o treino do Transformer TabNet...
epoch 0  | loss: 0.7475  | Treino_accuracy: 0.68834 | Validacao_accuracy: 0.68155 |  0:00:19s
epoch 1  | loss: 0.67794 | Treino_accuracy: 0.69052 | Validacao_accuracy: 0.68695 |  0:00:39s
epoch 2  | loss: 0.66799 | Treino_accuracy: 0.69426 | Validacao_accuracy: 0.69095 |  0:00:58s
epoch 3  | loss: 0.66271 | Treino_accuracy: 0.69345 | Validacao_accuracy: 0.6896  |  0:01:16s
epoch 4  | loss: 0.66127 | Treino_accuracy: 0.69545 | Validacao_accuracy: 0.69045 |  0:01:34s
epoch 5  | loss: 0.66088 | Treino_accuracy: 0.69774 | Validacao_accuracy: 0.6924  |  0:01:52s
epoch 6  | loss: 0.6587  | Treino_accuracy: 0.69485 | Validacao_accuracy: 0.69215 |  0:02:10s
epoch 7  | loss: 0.65744 | Treino_accuracy: 0.6975  | Validacao_accuracy: 0.68965 |  0:02:28s
epoch 8  | loss: 0.65567 | Treino_accuracy: 0.6963  | Validacao_accuracy: 0.69095 |  0:02:46s
epoch 9  | loss: 0.65384 | Treino_accuracy: 0.6977  | Validacao_accuracy: 0.6908  |  0:03:05s
epoch 10 | loss:

c:\Users\jluck\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning:

Best weights from best epoch are automatically used!



epoch 0  | loss: 0.75966 | Treino_accuracy: 0.67422 | Validacao_accuracy: 0.6708  |  0:00:16s
epoch 1  | loss: 0.68912 | Treino_accuracy: 0.68521 | Validacao_accuracy: 0.68055 |  0:00:33s
epoch 2  | loss: 0.6722  | Treino_accuracy: 0.69096 | Validacao_accuracy: 0.6861  |  0:00:50s
epoch 3  | loss: 0.66831 | Treino_accuracy: 0.68878 | Validacao_accuracy: 0.68505 |  0:01:07s
epoch 4  | loss: 0.6624  | Treino_accuracy: 0.69574 | Validacao_accuracy: 0.6903  |  0:01:24s
epoch 5  | loss: 0.66001 | Treino_accuracy: 0.69468 | Validacao_accuracy: 0.68795 |  0:01:41s
epoch 6  | loss: 0.65853 | Treino_accuracy: 0.69026 | Validacao_accuracy: 0.685   |  0:01:57s
epoch 7  | loss: 0.65667 | Treino_accuracy: 0.69192 | Validacao_accuracy: 0.68545 |  0:02:14s
epoch 8  | loss: 0.65438 | Treino_accuracy: 0.69389 | Validacao_accuracy: 0.6854  |  0:02:31s
epoch 9  | loss: 0.65385 | Treino_accuracy: 0.69274 | Validacao_accuracy: 0.6847  |  0:02:48s
epoch 10 | loss: 0.65254 | Treino_accuracy: 0.69331 | Valida

c:\Users\jluck\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning:

Best weights from best epoch are automatically used!



epoch 0  | loss: 0.76959 | Treino_accuracy: 0.67215 | Validacao_accuracy: 0.67075 |  0:00:16s
epoch 1  | loss: 0.68522 | Treino_accuracy: 0.68329 | Validacao_accuracy: 0.6814  |  0:00:33s
epoch 2  | loss: 0.66781 | Treino_accuracy: 0.69131 | Validacao_accuracy: 0.6885  |  0:00:50s
epoch 3  | loss: 0.66031 | Treino_accuracy: 0.69565 | Validacao_accuracy: 0.6926  |  0:01:08s
epoch 4  | loss: 0.65807 | Treino_accuracy: 0.69302 | Validacao_accuracy: 0.6913  |  0:01:26s
epoch 5  | loss: 0.65714 | Treino_accuracy: 0.69416 | Validacao_accuracy: 0.69175 |  0:01:43s
epoch 6  | loss: 0.65367 | Treino_accuracy: 0.6976  | Validacao_accuracy: 0.69445 |  0:02:00s
epoch 7  | loss: 0.65221 | Treino_accuracy: 0.69672 | Validacao_accuracy: 0.6963  |  0:02:17s
epoch 8  | loss: 0.65119 | Treino_accuracy: 0.697   | Validacao_accuracy: 0.6938  |  0:02:34s
epoch 9  | loss: 0.64966 | Treino_accuracy: 0.69491 | Validacao_accuracy: 0.6935  |  0:02:53s
epoch 10 | loss: 0.64949 | Treino_accuracy: 0.69875 | Valida

c:\Users\jluck\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning:

Best weights from best epoch are automatically used!



### Passo 9 - Testar modelos

In [9]:
# Modelos sem normalização
resultado_arvore = arvore.predict(x_test)
resultado_xgb = modelo_xgb.predict(x_test)

# Modelos com normalização
resultado_arvore_norm = arvore_norm.predict(x_test_norm)
resultado_xgb_norm = modelo_xgb_norm.predict(x_test_norm)

# Modelos com estratificação
resultado_arvore_s = arvore_s.predict(x_test_s)
resultado_xgb_s = modelo_xgb_s.predict(x_test_s)

# 1. Previsão sem normalização (Requer .values)
resultado_tabnet = modelo_tabnet.predict(x_test.values)

# 2. Previsão com normalização
resultado_tabnet_norm = modelo_tabnet_norm.predict(x_test_norm)

# 3. Previsão com estratificação (Requer .values)
resultado_tabnet_s = modelo_tabnet_s.predict(x_test_s.values)


### Passo 10 - Analisar resultados

In [10]:
# Acurácia
acc_arvore = accuracy_score(y_test, resultado_arvore)
acc_xgb = accuracy_score(y_test, resultado_xgb)
acc_tabnet = accuracy_score(y_test, resultado_tabnet)

acc_arvore_norm = accuracy_score(y_test, resultado_arvore_norm)
acc_xgb_norm = accuracy_score(y_test, resultado_xgb_norm)
acc_tabnet_norm = accuracy_score(y_test, resultado_tabnet_norm)

acc_arvore_s = accuracy_score(y_test_s, resultado_arvore_s)
acc_xgb_s = accuracy_score(y_test_s, resultado_xgb_s)
acc_tabnet_s = accuracy_score(y_test_s, resultado_tabnet_s)

# F1 Score
f1_arvore = f1_score(y_test, resultado_arvore, average='weighted')
f1_xgb = f1_score(y_test, resultado_xgb, average='weighted')
f1_tabnet = f1_score(y_test, resultado_tabnet, average='weighted')

f1_arvore_norm = f1_score(y_test, resultado_arvore_norm, average='weighted')
f1_xgb_norm = f1_score(y_test, resultado_xgb_norm, average='weighted')
f1_tabnet_norm = f1_score(y_test, resultado_tabnet_norm, average='weighted')

f1_arvore_s = f1_score(y_test_s, resultado_arvore_s, average='weighted')
f1_xgb_s = f1_score(y_test_s, resultado_xgb_s, average='weighted')
f1_tabnet_s = f1_score(y_test_s, resultado_tabnet_s, average='weighted')

# Precision
precision_arvore = precision_score(y_test, resultado_arvore, average='weighted')
precision_xgb = precision_score(y_test, resultado_xgb, average='weighted')
precision_tabnet = precision_score(y_test, resultado_tabnet, average='weighted')

precision_arvore_norm = precision_score(y_test, resultado_arvore_norm, average='weighted')
precision_xgb_norm = precision_score(y_test, resultado_xgb_norm, average='weighted')
precision_tabnet_norm = precision_score(y_test, resultado_tabnet_norm, average='weighted')

precision_arvore_s = precision_score(y_test_s, resultado_arvore_s, average='weighted')
precision_xgb_s = precision_score(y_test_s, resultado_xgb_s, average='weighted')
precision_tabnet_s = precision_score(y_test_s, resultado_tabnet_s, average='weighted')

# Recall
recall_arvore = recall_score(y_test, resultado_arvore, average='weighted')
recall_xgb = recall_score(y_test, resultado_xgb, average='weighted')
recall_tabnet = recall_score(y_test, resultado_tabnet, average='weighted')

recall_arvore_norm = recall_score(y_test, resultado_arvore_norm, average='weighted')
recall_xgb_norm = recall_score(y_test, resultado_xgb_norm, average='weighted')
recall_tabnet_norm = recall_score(y_test, resultado_tabnet_norm, average='weighted')

recall_arvore_s = recall_score(y_test_s, resultado_arvore_s, average='weighted')
recall_xgb_s = recall_score(y_test_s, resultado_xgb_s, average='weighted')
recall_tabnet_s = recall_score(y_test_s, resultado_tabnet_s, average='weighted')


# --- 2. DATAFRAMES PARA OS GRÁFICOS ---

# DataFrame de Normalização
dados_metricas_norm = {
    'Modelo': ['Random Forest', 'XGBoost', 'Transformer', 'Random Forest', 'XGBoost', 'Transformer'],
    'Normalização': ['Sem', 'Sem', 'Sem', 'Com', 'Com', 'Com'], 
    'Acurácia': [acc_arvore, acc_xgb, acc_tabnet, acc_arvore_norm, acc_xgb_norm, acc_tabnet_norm],
    'F1 Score': [f1_arvore, f1_xgb, f1_tabnet, f1_arvore_norm, f1_xgb_norm, f1_tabnet_norm],
    'Precision': [precision_arvore, precision_xgb, precision_tabnet, precision_arvore_norm, precision_xgb_norm, precision_tabnet_norm],
    'Recall': [recall_arvore, recall_xgb, recall_tabnet, recall_arvore_norm, recall_xgb_norm, recall_tabnet_norm]
}
df_metricas_norm = pd.DataFrame(dados_metricas_norm)

# DataFrame de Estratificação
dados_metricas_estrat = {
    'Modelo': ['Random Forest', 'XGBoost', 'Transformer', 'Random Forest', 'XGBoost', 'Transformer'],
    'Estratificação': ['Sem', 'Sem', 'Sem', 'Com', 'Com', 'Com'], 
    'Acurácia': [acc_arvore, acc_xgb, acc_tabnet, acc_arvore_s, acc_xgb_s, acc_tabnet_s],
    'F1 Score': [f1_arvore, f1_xgb, f1_tabnet, f1_arvore_s, f1_xgb_s, f1_tabnet_s],
    'Precision': [precision_arvore, precision_xgb, precision_tabnet, precision_arvore_s, precision_xgb_s, precision_tabnet_s],
    'Recall': [recall_arvore, recall_xgb, recall_tabnet, recall_arvore_s, recall_xgb_s, recall_tabnet_s]
}
df_metricas_estrat = pd.DataFrame(dados_metricas_estrat)


# CONFIGURAÇÃO MÁGICA: Dicionário para remover totalmente os fundos
layout_sem_fundo = dict(
    paper_bgcolor='rgba(0,0,0,0)', 
    plot_bgcolor='rgba(0,0,0,0)',    
    font_color = 'gray'
)

# --- 3. GERAR GRÁFICOS (NORMALIZAÇÃO) ---
grafico_acc_norm = px.bar(
    df_metricas_norm, x='Modelo', y='Acurácia', color='Normalização', barmode='group',
    title='Comparação de Acurácia (Impacto da Normalização)', text_auto='.4f'
)
grafico_acc_norm.update_layout(layout_sem_fundo)

grafico_f1_norm = px.bar(
    df_metricas_norm, x='Modelo', y='F1 Score', color='Normalização', barmode='group',
    title='Comparação de F1 Score (Impacto da Normalização)', text_auto='.4f'
)
grafico_f1_norm.update_layout(layout_sem_fundo)

# --- 4. GERAR GRÁFICOS (ESTRATIFICAÇÃO) ---
grafico_acc_estrat = px.bar(
    df_metricas_estrat, x='Modelo', y='Acurácia', color='Estratificação', barmode='group',
    title='Comparação de Acurácia (Impacto da Estratificação)', text_auto='.4f'
)
grafico_acc_estrat.update_layout(layout_sem_fundo)

grafico_f1_estrat = px.bar(
    df_metricas_estrat, x='Modelo', y='F1 Score', color='Estratificação', barmode='group',
    title='Comparação de F1 Score (Impacto da Estratificação)', text_auto='.4f'
)
grafico_f1_estrat.update_layout(layout_sem_fundo)

# --- 5. EXIBIR OS GRÁFICOS ---
print("--- GRÁFICOS DE NORMALIZAÇÃO ---")
grafico_acc_norm.show()
grafico_f1_norm.show()

print("\n--- GRÁFICOS DE ESTRATIFICAÇÃO ---")
grafico_acc_estrat.show()
grafico_f1_estrat.show()

--- GRÁFICOS DE NORMALIZAÇÃO ---



--- GRÁFICOS DE ESTRATIFICAÇÃO ---


### Passo 11 - Realizar previsões

In [11]:
# Importando a tabela de novos clientes
tabela_novos_clientes = pd.read_csv("novos_clientes.csv")

# Codificação de variáveis categóricas na tabela de novos clientes
for coluna in tabela_novos_clientes.columns:
    if tabela_novos_clientes[coluna].dtype == 'object':
        tabela_novos_clientes[coluna] = encoders[coluna].transform(tabela_novos_clientes[coluna])

# Removendo a coluna ID da tabela de novos clientes
if "id_cliente" in tabela_novos_clientes.columns:
    tabela_novos_clientes = tabela_novos_clientes.drop("id_cliente", axis=1)

# Previsões para novos clientes
previsao_arvore = encoders['score_credito'].inverse_transform(arvore.predict(tabela_novos_clientes))

# Exibir a tabela de novos clientes e as previsões
display(tabela_novos_clientes)
display(list(previsao_arvore))

,mes,idade,profissao,salario_anual,num_contas,num_cartoes,juros_emprestimo,num_emprestimos,dias_atraso,num_pagamentos_atrasados,...,taxa_uso_credito,idade_historico_credito,investimento_mensal,comportamento_pagamento,saldo_final_mes,emprestimo_carro,emprestimo_casa,emprestimo_pessoal,emprestimo_credito,emprestimo_estudantil
0,1,31.0,5,19300.340,6.0,7.0,17.0,5.0,52.0,19.0,...,29.934186,218.0,44.50951,4,312.487689,1,1,0,0,0
1,4,32.0,0,12600.445,5.0,5.0,10.0,3.0,25.0,18.0,...,28.819407,12.0,0.00000,5,300.994163,0,0,0,0,1
2,2,48.0,5,20787.690,8.0,6.0,14.0,7.0,24.0,14.0,...,34.235853,215.0,0.00000,3,345.081577,0,1,0,1,0


['Poor', 'Poor', 'Standard']

### Exportar modelo Random Florest para JavaScript
Isso permitirá utilizá-lo na interface web

In [12]:
!pip install m2cgen

import m2cgen as m2c

# 1. Traduz o modelo do scikit-learn para uma função JavaScript pura
# Substitua 'arvore' pelo nome da variável do seu melhor modelo Random Forest
codigo_js = m2c.export_to_javascript(arvore)

# 2. Salva esse código num ficheiro
with open("modelo_rf.js", "w") as f:
    f.write(codigo_js)

print("Modelo exportado com sucesso para modelo_rf.js!")

Modelo exportado com sucesso para modelo_rf.js!


### Exportar Parâmetros
Isto permitirá a correta utilização do modelo na interface web

In [16]:
import json

# Construir o dicionário base para a exportação
parametros_exportacao["encoders"][coluna] = {
    str(classe): int(idx) for idx, classe in enumerate(encoder.classes_)
}

# Extrair os dicionários ("De -> Para") fixos do LabelEncoder 
for coluna, encoder in encoders.items():
    parametros_exportacao["encoders"][coluna] = {
        str(classe): int(idx) for idx, classe in enumerate(encoder.classes_)
    }

# Salvar o mapeamento em um arquivo .json
with open("parametros_preprocessing.json", "w", encoding="utf-8") as f:
    json.dump(parametros_exportacao, f, ensure_ascii=False, indent=4)

print("Parâmetros estáticos exportados com sucesso!")

Parâmetros estáticos exportados com sucesso!


### Como melhorar o modelo:

- Gridsearch: Buscar melhor combinação de parâmetros do modelo para o cenário;
- Tratamento de Dados: Quanto melhor a qualidade dos dados, mais preciso será o modelo;
- Dados Representativos: Particionar os dados de forma aos conjuntos de testes e treino possuir boa distribuição de dados;
- Quantidade de dados: Quanto mais dados para testar melhor, porém o conjunto de treino não pode ser muito pequeno patra não enviesar os resultados. Mantenha a faixa 60-80/40-20.

### Os trechos a seguir buscam a melhor combinação de parâmetros para os algoritmos

In [14]:
# 1. Definir o dicionário de opções (os botões que a IA vai girar)
parametros_rf = {
    'n_estimators': [i for i in range(1,1001)],         # Testa florestas com 100, 200 ou 300 árvores
    'max_depth': [i for i in range(1,51)],         # Testa diferentes limites de profundidade
    'min_samples_split': [i for i in range(1,51)],         # Testa regras de separação
    'min_samples_leaf': [i for i in range(1,51)]            # Mínimo de clientes no fim da árvore
}

# 2. Criar o "Afinador" automático
# n_iter=10 significa que ele vai sortear e testar 10 combinações diferentes dessas opções acima
# cv=3 significa Validação Cruzada (ele testa cada combinação 3 vezes para ter certeza)
# n_jobs=-1 faz o seu computador usar TODOS os núcleos de processamento para terminar mais rápido
afinador_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=parametros_rf,
    n_iter=10, 
    cv=3, 
    random_state=42,
    n_jobs=-1 
)

# 3. Treinar o Afinador (Atenção: Esta célula vai demorar alguns minutos a rodar!)
print("A procurar a melhor configuração para o Random Forest...")
afinador_rf.fit(x_train_norm, y_train_s)

# 4. Exibir o que ele descobriu
print("\nA busca terminou!")
print("Melhores botões escolhidos:", afinador_rf.best_params_)

# 5. Salvar o modelo campeão e fazer as previsões
melhor_arvore = afinador_rf.best_estimator_
resultado_melhor_arvore = melhor_arvore.predict(x_test_norm)

# 6. Avaliar se melhorou a Acurácia
acc_arvore_tunada = accuracy_score(y_test_s, resultado_melhor_arvore)
print(f"\nAcurácia da nova Árvore Otimizada: {acc_arvore_tunada:.4f}")

A procurar a melhor configuração para o Random Forest...

A busca terminou!
Melhores botões escolhidos: {'n_estimators': 37, 'min_samples_split': 6, 'min_samples_leaf': 36, 'max_depth': 23}

Acurácia da nova Árvore Otimizada: 0.5317


In [15]:
# Dicionário de botões do XGBoost
parametros_xgb = {
    'n_estimators': [i for i in range(50,1001,50)],
    'max_depth': [i for i in range (1,16,1)],             # XGBoost prefere árvores mais rasas que o RF
    'learning_rate': [i/100 for i in range(1,31,1)],      # O botão mágico do Boosting
    'subsample': [i/100 for i in range(1,101,1)]          # Fração de clientes sorteados por árvore
}

afinador_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42),
    param_distributions=parametros_xgb,
    n_iter=10, cv=3, random_state=42, n_jobs=-1
)

print("A procurar a melhor configuração para o XGB...")
afinador_xgb.fit(x_train_norm, y_train_s)

print("Melhores botões escolhidos:", afinador_xgb.best_params_)
melhor_xgb = afinador_xgb.best_estimator_
resultado_melhor_xgb = melhor_xgb.predict(x_test_norm)

# 6. Avaliar se melhorou a Acurácia
acc_xgb_tunada = accuracy_score(y_test_s, resultado_melhor_xgb)
print(f"\nAcurácia da novo XGB Otimizado: {acc_arvore_tunada:.4f}")

A procurar a melhor configuração para o XGB...
Melhores botões escolhidos: {'subsample': 0.59, 'n_estimators': 1000, 'max_depth': 1, 'learning_rate': 0.05}

Acurácia da novo XGB Otimizado: 0.5317
